In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import glob
import os
from datetime import datetime

In [4]:
# Ver cuántos archivos JSON tienes
json_files = glob.glob('../../Limpieza/data/resultados')
print(f"📊 Encontrados: {len(json_files)} archivos de resultados")

📊 Encontrados: 1121 archivos de resultados


In [13]:
# Configurar estilo de gráficos
plt.style.use('default')
sns.set_palette("husl")

In [14]:
class GeneradorMapasCalor:
    """Clase principal para generar todos los mapas de calor"""

    def __init__(self, ruta_resultados='resultados', debug=True):
        self.ruta = ruta_resultados
        self.debug = debug
        self.datos_cargados = None
        self.mapas_generados = []

    def log(self, mensaje):
        """Función para logging con timestamp"""
        if self.debug:
            timestamp = datetime.now().strftime("%H:%M:%S")
            print(f"[{timestamp}] {mensaje}")

    def cargar_datos(self):
        """Carga y valida todos los archivos JSON"""
        self.log("🔄 Cargando datos desde archivos JSON...")

        # Buscar archivos JSON
        json_files = glob.glob(os.path.join(self.ruta, '*_results.json'))

        if len(json_files) == 0:
            print("❌ No se encontraron archivos JSON en la carpeta")
            return False

        datos = []
        errores = 0

        for json_file in json_files:
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    datos.append(data)
            except Exception as e:
                errores += 1
                if self.debug:
                    self.log(f"Error leyendo {os.path.basename(json_file)}: {str(e)}")

        self.datos_cargados = datos
        self.log(f"✅ Cargados {len(datos)} archivos correctamente ({errores} errores)")

        return len(datos) > 0

    def generar_heatmap_velocidades_por_tecnologia(self, metric='velocidad_media'):
        """Genera mapa de calor: Velocidades por Tecnología y Municipio"""
        self.log(f"📊 Generando heatmap: {metric} por tecnología...")

        # Recopilar datos
        data_rows = []
        for data in self.datos_cargados:
            municipio = data.get('municipio', 'Unknown')

            if 'tecnologias' in data and data['tecnologias'].get('trends'):
                for tech, trends in data['tecnologias']['trends'].items():
                    if metric in trends and trends[metric] is not None:
                        data_rows.append({
                            'municipio': municipio,
                            'tecnologia': tech,
                            'valor': trends[metric]
                        })

        if len(data_rows) == 0:
            self.log(f"⚠️ No hay datos para {metric}")
            return None

        # Crear DataFrame y tabla pivote
        df = pd.DataFrame(data_rows)
        pivot_table = df.pivot_table(
            index='municipio',
            columns='tecnologia',
            values='valor',
            fill_value=0
        )

        # Configurar el mapa de calor
        plt.figure(figsize=(15, max(8, len(pivot_table.index) * 0.4)))

        if 'r2' in metric.lower():
            cmap, vmin, vmax, fmt = 'RdYlGn', 0, 1, '.3f'
        elif 'velocidad' in metric.lower():
            cmap, vmin, vmax, fmt = 'viridis', None, None, '.1f'
        else:
            cmap, vmin, vmax, fmt = 'plasma', None, None, '.0f'

        sns.heatmap(pivot_table,
                    annot=True,
                    cmap=cmap,
                    fmt=fmt,
                    cbar_kws={'shrink': .8},
                    vmin=vmin,
                    vmax=vmax)

        plt.title(f'🔥 {metric.replace("_", " ").title()} por Municipio y Tecnología',
                  fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Tecnología', fontsize=12)
        plt.ylabel('Municipio', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, f'heatmap_tecnologias_{metric}.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_ranking_velocidades(self, speed_type='bajada', metric='media'):
        """Genera ranking de municipios por velocidad"""
        self.log(f"📊 Generando ranking: velocidad {speed_type} - {metric}...")

        # Recopilar datos
        data_rows = []
        for data in self.datos_cargados:
            municipio = data.get('municipio', 'Unknown')

            if ('velocidades' in data and
                data['velocidades'].get('trends') and
                speed_type in data['velocidades']['trends']):

                speed_data = data['velocidades']['trends'][speed_type]
                if metric in speed_data and speed_data[metric] is not None:
                    data_rows.append({
                        'municipio': municipio,
                        'valor': speed_data[metric]
                    })

        if len(data_rows) == 0:
            self.log(f"⚠️ No hay datos para velocidad {speed_type} - {metric}")
            return None

        # Crear DataFrame y ordenar
        df = pd.DataFrame(data_rows)
        df = df.sort_values('valor', ascending=False)

        # Tomar solo los primeros 20 para mejor visualización
        if len(df) > 20:
            df = df.head(20)
            titulo_extra = " (Top 20)"
        else:
            titulo_extra = ""

        # Crear el heatmap
        plt.figure(figsize=(12, max(8, len(df) * 0.4)))

        # Crear matriz para el heatmap
        heatmap_data = df['valor'].values.reshape(-1, 1)

        # Configurar colores según métrica
        if metric == 'r2':
            cmap, vmin, vmax, fmt = 'RdYlGn', 0, 1, '.3f'
        elif metric == 'tendencia':
            cmap, vmin, vmax, fmt = 'RdBu_r', None, None, '.4f'
        else:  # media
            cmap, vmin, vmax, fmt = 'viridis', None, None, '.1f'

        sns.heatmap(heatmap_data,
                    annot=True,
                    cmap=cmap,
                    fmt=fmt,
                    yticklabels=df['municipio'].values,
                    xticklabels=[f'Velocidad {speed_type.title()}'],
                    cbar_kws={'shrink': .8},
                    vmin=vmin,
                    vmax=vmax)

        plt.title(f'🏆 Ranking: Velocidad {speed_type.title()} - {metric.title()}{titulo_extra}',
                  fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('')
        plt.ylabel('Municipio', fontsize=12)
        plt.yticks(rotation=0)
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, f'ranking_velocidades_{speed_type}_{metric}.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_evolucion_temporal(self):
        """Genera mapa de calor de evolución temporal"""
        self.log("📊 Generando evolución temporal...")

        # Recopilar datos temporales
        temporal_data = []
        for data in self.datos_cargados:
            municipio = data.get('municipio', 'Unknown')

            if ('velocidades' in data and
                data['velocidades'].get('stats')):

                stats = data['velocidades']['stats']
                for stat_row in stats:
                    temporal_data.append({
                        'municipio': municipio,
                        'trimestre': stat_row.get('TRIMESTRE', 0),
                        'velocidad_bajada': stat_row.get('VELOCIDAD_BAJADA_MEAN', 0)
                    })

        if len(temporal_data) == 0:
            self.log("⚠️ No hay datos temporales disponibles")
            return None

        # Crear DataFrame y tabla pivote
        df = pd.DataFrame(temporal_data)
        pivot_table = df.pivot_table(
            index='municipio',
            columns='trimestre',
            values='velocidad_bajada',
            fill_value=0
        )

        # Limitar municipios si son muchos
        if len(pivot_table.index) > 25:
            # Tomar municipios con mayor velocidad promedio
            avg_speeds = pivot_table.mean(axis=1).sort_values(ascending=False)
            pivot_table = pivot_table.loc[avg_speeds.head(25).index]

        # Crear el heatmap
        plt.figure(figsize=(16, max(8, len(pivot_table.index) * 0.4)))

        sns.heatmap(pivot_table,
                    annot=True,
                    cmap='viridis',
                    fmt='.1f',
                    cbar_kws={'shrink': .8, 'label': 'Velocidad Bajada (Mbps)'})

        plt.title('📈 Evolución Temporal: Velocidad de Bajada por Municipio y Trimestre',
                  fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Trimestre', fontsize=12)
        plt.ylabel('Municipio', fontsize=12)
        plt.xticks(rotation=0)
        plt.yticks(rotation=0)
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, 'evolucion_temporal_velocidades.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_comparativo_tecnologias(self):
        """Genera mapa comparativo de tecnologías"""
        self.log("📊 Generando comparativo de tecnologías...")

        # Recopilar datos por tecnología
        tech_data = {}
        for data in self.datos_cargados:
            if 'tecnologias' in data and data['tecnologias'].get('trends'):
                for tech, trends in data['tecnologias']['trends'].items():
                    if tech not in tech_data:
                        tech_data[tech] = {
                            'velocidad_media': [],
                            'accesos_media': [],
                            'r2_velocidad': [],
                            'r2_accesos': []
                        }

                    # Recopilar métricas
                    for metric in ['velocidad_media', 'accesos_media']:
                        if metric in trends and trends[metric] is not None:
                            tech_data[tech][metric].append(trends[metric])

                    if 'r2_velocidad' in trends and trends['r2_velocidad'] is not None:
                        tech_data[tech]['r2_velocidad'].append(trends['r2_velocidad'])
                    if 'r2' in trends and trends['r2'] is not None:
                        tech_data[tech]['r2_accesos'].append(trends['r2'])

        if len(tech_data) == 0:
            self.log("⚠️ No hay datos de tecnologías disponibles")
            return None

        # Calcular promedios
        comparison_data = []
        for tech, metrics in tech_data.items():
            row = {'tecnologia': tech}
            for metric_name, values in metrics.items():
                if values:
                    row[metric_name] = np.mean(values)
                else:
                    row[metric_name] = 0
            comparison_data.append(row)

        # Crear DataFrame
        df = pd.DataFrame(comparison_data)

        # Preparar datos para heatmap
        metric_columns = ['velocidad_media', 'accesos_media', 'r2_velocidad', 'r2_accesos']
        heatmap_data = df[metric_columns].set_index(df['tecnologia'])

        # Normalizar datos (0-1) para comparación visual
        heatmap_normalized = heatmap_data.div(heatmap_data.max(), axis=1).fillna(0)

        # Crear el heatmap
        plt.figure(figsize=(12, max(6, len(heatmap_data.index) * 0.6)))

        sns.heatmap(heatmap_normalized,
                    annot=heatmap_data,  # Valores reales
                    cmap='RdYlGn',
                    fmt='.2f',
                    cbar_kws={'shrink': .8, 'label': 'Valor Normalizado (0-1)'})

        plt.title('⚖️ Comparación de Métricas por Tecnología (Valores Promedio)',
                  fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Métrica', fontsize=12)
        plt.ylabel('Tecnología', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, 'comparativo_tecnologias.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_resumen_estadistico(self):
        """Genera un resumen estadístico visual"""
        self.log("📊 Generando resumen estadístico...")

        # Recopilar estadísticas generales
        velocidades_bajada = []
        velocidades_subida = []
        municipios_analizados = []

        for data in self.datos_cargados:
            municipio = data.get('municipio', 'Unknown')
            municipios_analizados.append(municipio)

            if ('velocidades' in data and
                data['velocidades'].get('trends')):

                vel_trends = data['velocidades']['trends']
                if 'bajada' in vel_trends and 'media' in vel_trends['bajada']:
                    velocidades_bajada.append(vel_trends['bajada']['media'])
                if 'subida' in vel_trends and 'media' in vel_trends['subida']:
                    velocidades_subida.append(vel_trends['subida']['media'])

        # Crear gráfico de resumen
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

        # 1. Histograma velocidades bajada
        if velocidades_bajada:
            ax1.hist(velocidades_bajada, bins=20, color='blue', alpha=0.7, edgecolor='black')
            ax1.set_title('Distribución: Velocidades de Bajada')
            ax1.set_xlabel('Velocidad (Mbps)')
            ax1.set_ylabel('Frecuencia')
            ax1.grid(True, alpha=0.3)

        # 2. Histograma velocidades subida
        if velocidades_subida:
            ax2.hist(velocidades_subida, bins=20, color='green', alpha=0.7, edgecolor='black')
            ax2.set_title('Distribución: Velocidades de Subida')
            ax2.set_xlabel('Velocidad (Mbps)')
            ax2.set_ylabel('Frecuencia')
            ax2.grid(True, alpha=0.3)

        # 3. Scatter plot bajada vs subida
        if velocidades_bajada and velocidades_subida:
            min_len = min(len(velocidades_bajada), len(velocidades_subida))
            ax3.scatter(velocidades_bajada[:min_len], velocidades_subida[:min_len],
                       alpha=0.6, color='purple')
            ax3.set_title('Correlación: Bajada vs Subida')
            ax3.set_xlabel('Velocidad Bajada (Mbps)')
            ax3.set_ylabel('Velocidad Subida (Mbps)')
            ax3.grid(True, alpha=0.3)

        # 4. Estadísticas básicas
        stats_text = f"""
RESUMEN ESTADÍSTICO

📊 Municipios analizados: {len(municipios_analizados)}

🔽 Velocidad Bajada:
   • Promedio: {np.mean(velocidades_bajada):.1f} Mbps
   • Mediana: {np.median(velocidades_bajada):.1f} Mbps
   • Min: {np.min(velocidades_bajada):.1f} Mbps
   • Max: {np.max(velocidades_bajada):.1f} Mbps

🔼 Velocidad Subida:
   • Promedio: {np.mean(velocidades_subida):.1f} Mbps
   • Mediana: {np.median(velocidades_subida):.1f} Mbps
   • Min: {np.min(velocidades_subida):.1f} Mbps
   • Max: {np.max(velocidades_subida):.1f} Mbps
        """

        ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.5))
        ax4.set_xlim(0, 1)
        ax4.set_ylim(0, 1)
        ax4.axis('off')

        plt.suptitle('📊 Resumen Estadístico General', fontsize=16, fontweight='bold')
        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(self.ruta, 'resumen_estadistico.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=300)
        plt.close()

        self.mapas_generados.append(plot_path)
        self.log(f"✅ Guardado: {os.path.basename(plot_path)}")

        return plot_path

    def generar_todos_los_mapas(self):
        """Función principal que genera todos los mapas de calor"""
        print("🔥 INICIANDO GENERACIÓN COMPLETA DE MAPAS DE CALOR")
        print("="*60)

        # Cargar datos
        if not self.cargar_datos():
            print("❌ Error: No se pudieron cargar los datos")
            return []

        print(f"\n🎯 Generando mapas desde: {os.path.abspath(self.ruta)}")

        # Lista de funciones a ejecutar
        generadores = [
            ("Velocidades por Tecnología", lambda: self.generar_heatmap_velocidades_por_tecnologia('velocidad_media')),
            ("Calidad R² por Tecnología", lambda: self.generar_heatmap_velocidades_por_tecnologia('r2_velocidad')),
            ("Ranking Velocidades Bajada", lambda: self.generar_ranking_velocidades('bajada', 'media')),
            ("Ranking Velocidades Subida", lambda: self.generar_ranking_velocidades('subida', 'media')),
            ("Evolución Temporal", self.generar_evolucion_temporal),
            ("Comparativo Tecnologías", self.generar_comparativo_tecnologias),
            ("Resumen Estadístico", self.generar_resumen_estadistico)
        ]

        # Ejecutar cada generador
        for i, (nombre, funcion) in enumerate(generadores, 1):
            print(f"\n📊 {i}/{len(generadores)}. {nombre}...")
            try:
                resultado = funcion()
                if resultado:
                    print(f"   ✅ Completado")
                else:
                    print(f"   ⚠️ Sin datos suficientes")
            except Exception as e:
                print(f"   ❌ Error: {str(e)}")

        # Resumen final
        print(f"\n🎉 GENERACIÓN COMPLETADA!")
        print("="*60)
        print(f"📊 Total de mapas generados: {len(self.mapas_generados)}")

        if self.mapas_generados:
            print(f"\n📁 ARCHIVOS GENERADOS:")
            for i, mapa in enumerate(self.mapas_generados, 1):
                print(f"   {i:2d}. {os.path.basename(mapa)}")

            print(f"\n📂 Ubicación: {os.path.abspath(self.ruta)}")
            print(f"\n💡 Los mapas están listos para revisar!")

        return self.mapas_generados


In [15]:

def ejecutar_mapas_calor(carpeta='../../Limpieza/data/resultados'):
    """
    Función simple para ejecutar todo el sistema
    """
    # Crear el generador
    generador = GeneradorMapasCalor(carpeta, debug=True)

    # Generar todos los mapas
    mapas = generador.generar_todos_los_mapas()

    return mapas

In [16]:
def verificar_ruta_datos():
    """Verifica que la ruta sea correcta"""
    ruta = '../../Limpieza/data/resultados'
    json_files = glob.glob(f'{ruta}/*_results.json')

    print(f"🔍 Verificando ruta: {os.path.abspath(ruta)}")
    print(f"📊 Archivos JSON encontrados: {len(json_files)}")

    if len(json_files) > 0:
        print(f"✅ Ruta correcta! Ejemplo de archivo: {os.path.basename(json_files[0])}")
        return True
    else:
        print(f"❌ No se encontraron archivos JSON en esa ruta")
        print(f"💡 Verifica que la ruta sea correcta")
        return False


In [17]:
verificar_ruta_datos()

🔍 Verificando ruta: /home/kingkold/data-projects/InternetAccessColombia/InternetAccessColombia-DataCleaning/Limpieza/data/resultados
📊 Archivos JSON encontrados: 1121
✅ Ruta correcta! Ejemplo de archivo: BOLÍVAR_ARENAL_results.json


True

In [18]:
mapas_generados = ejecutar_mapas_calor('../../Limpieza/data/resultados')

🔥 INICIANDO GENERACIÓN COMPLETA DE MAPAS DE CALOR
[01:26:40] 🔄 Cargando datos desde archivos JSON...
[01:26:41] ✅ Cargados 1121 archivos correctamente (0 errores)

🎯 Generando mapas desde: /home/kingkold/data-projects/InternetAccessColombia/InternetAccessColombia-DataCleaning/Limpieza/data/resultados

📊 1/7. Velocidades por Tecnología...
[01:26:41] 📊 Generando heatmap: velocidad_media por tecnología...


/tmp/ipykernel_61197/906087099.py:100: UserWarning: Glyph 128293 (\N{FIRE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_61197/906087099.py:104: UserWarning: Glyph 128293 (\N{FIRE}) missing from font(s) DejaVu Sans.
  plt.savefig(plot_path, bbox_inches='tight', dpi=300)


[01:27:34] ✅ Guardado: heatmap_tecnologias_velocidad_media.png
   ✅ Completado

📊 2/7. Calidad R² por Tecnología...
[01:27:34] 📊 Generando heatmap: r2_velocidad por tecnología...


/tmp/ipykernel_61197/906087099.py:100: UserWarning: Glyph 128293 (\N{FIRE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_61197/906087099.py:104: UserWarning: Glyph 128293 (\N{FIRE}) missing from font(s) DejaVu Sans.
  plt.savefig(plot_path, bbox_inches='tight', dpi=300)


[01:28:25] ✅ Guardado: heatmap_tecnologias_r2_velocidad.png
   ✅ Completado

📊 3/7. Ranking Velocidades Bajada...
[01:28:25] 📊 Generando ranking: velocidad bajada - media...


...

[01:28:26] ✅ Guardado: ranking_velocidades_bajada_media.png
   ✅ Completado

📊 4/7. Ranking Velocidades Subida...
[01:28:26] 📊 Generando ranking: velocidad subida - media...


/tmp/ipykernel_61197/906087099.py:176: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_61197/906087099.py:180: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig(plot_path, bbox_inches='tight', dpi=300)


[01:28:27] ✅ Guardado: ranking_velocidades_subida_media.png
   ✅ Completado

📊 5/7. Evolución Temporal...
[01:28:27] 📊 Generando evolución temporal...
   ❌ Error: 'str' object has no attribute 'get'

📊 6/7. Comparativo Tecnologías...
[01:28:27] 📊 Generando comparativo de tecnologías...
[01:28:27] ✅ Guardado: comparativo_tecnologias.png
   ✅ Completado

📊 7/7. Resumen Estadístico...
[01:28:27] 📊 Generando resumen estadístico...


/tmp/ipykernel_61197/906087099.py:411: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_61197/906087099.py:411: UserWarning: Glyph 128317 (\N{DOWN-POINTING SMALL RED TRIANGLE}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_61197/906087099.py:411: UserWarning: Glyph 128316 (\N{UP-POINTING SMALL RED TRIANGLE}) missing from font(s) DejaVu Sans Mono.
  plt.tight_layout()
/tmp/ipykernel_61197/906087099.py:411: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_61197/906087099.py:415: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans Mono.
  plt.savefig(plot_path, bbox_inches='tight', dpi=300)
/tmp/ipykernel_61197/906087099.py:415: UserWarning: Glyph 128317 (\N{DOWN-POINTING SMALL RED TRIANGLE}) missing from font(s) DejaVu Sans Mono.
  plt.savefig(plot_path, bbox_inches='tight', dpi=300)
/tmp/ipykernel_61197/906

[01:28:28] ✅ Guardado: resumen_estadistico.png
   ✅ Completado

🎉 GENERACIÓN COMPLETADA!
📊 Total de mapas generados: 6

📁 ARCHIVOS GENERADOS:
    1. heatmap_tecnologias_velocidad_media.png
    2. heatmap_tecnologias_r2_velocidad.png
    3. ranking_velocidades_bajada_media.png
    4. ranking_velocidades_subida_media.png
    5. comparativo_tecnologias.png
    6. resumen_estadistico.png

📂 Ubicación: /home/kingkold/data-projects/InternetAccessColombia/InternetAccessColombia-DataCleaning/Limpieza/data/resultados

💡 Los mapas están listos para revisar!
